In [13]:
import notebookutils
from pyspark.sql.functions import (
    col, when, lit, current_timestamp
)
from datetime import datetime

# ── Lakehouse paths ─────────────────────────────────────────
SILVER_PATH   = "abfss://bddf1263-676e-44e7-bf20-6d7b218dc2d2@onelake.dfs.fabric.microsoft.com/c49fa9c5-5ba8-4add-a7df-fa808a6fa426"
GOLD_PATH     = "abfss://bddf1263-676e-44e7-bf20-6d7b218dc2d2@onelake.dfs.fabric.microsoft.com/07bd3e33-6f0c-4dac-b2a4-b70dca6c6298"

GOLD_TABLE    = "gold_fact_transactions"
PIPELINE_NAME = "NB_09_Gold_FactTransactions"
BATCH_DATE    = datetime.now().strftime("%Y-%m-%d")

print(f"Gold Fact Transactions Pipeline")
print(f"Silver : {SILVER_PATH}")
print(f"Gold   : {GOLD_PATH}")
print(f"Started: {datetime.now()}")

StatementMeta(, 75aa9f49-bfb0-45ba-b525-ffdfe4a8231d, 15, Finished, Available, Finished, False)

Gold Fact Transactions Pipeline
Silver : abfss://bddf1263-676e-44e7-bf20-6d7b218dc2d2@onelake.dfs.fabric.microsoft.com/c49fa9c5-5ba8-4add-a7df-fa808a6fa426
Gold   : abfss://bddf1263-676e-44e7-bf20-6d7b218dc2d2@onelake.dfs.fabric.microsoft.com/07bd3e33-6f0c-4dac-b2a4-b70dca6c6298
Started: 2026-05-06 01:12:16.132351


In [14]:
# ── Read Silver tables using direct ABFSS paths ─────────────
SILVER_TABLES = f"{SILVER_PATH}/Tables/dbo"

df_transactions = spark.read.format("delta").load(
    f"{SILVER_TABLES}/silver_transactions")

df_merchants = spark.read.format("delta").load(
    f"{SILVER_TABLES}/silver_merchants")

df_cardholders = spark.read.format("delta").load(
    f"{SILVER_TABLES}/silver_cardholders")

df_fraud = spark.read.format("delta").load(
    f"{SILVER_TABLES}/silver_fraud_labels")

print(f"Transactions : {df_transactions.count():,}")
print(f"Merchants    : {df_merchants.count():,}")
print(f"Cardholders  : {df_cardholders.count():,}")
print(f"Fraud labels : {df_fraud.count():,}")

StatementMeta(, 75aa9f49-bfb0-45ba-b525-ffdfe4a8231d, 16, Finished, Available, Finished, False)

Transactions : 145,946
Merchants    : 2,000
Cardholders  : 5,000
Fraud labels : 2,800


In [15]:
# ── Slim down each table ────────────────────────────────────
merchants_slim = df_merchants.select(
    "merchant_id",
    "merchant_name",
    "category_code",
    "category_name",
    col("city").alias("merchant_city"),
    col("province").alias("merchant_province"),
    "acquiring_bank",
    col("risk_rating").alias("merchant_risk_rating"),
    col("status").alias("merchant_status")
)

cardholders_slim = df_cardholders.select(
    "cardholder_id",
    "issuing_bank",
    col("province").alias("cardholder_province"),
    "income_band",
    "credit_band",
    col("risk_rating").alias("cardholder_risk_rating"),
    col("status").alias("cardholder_status"),
    "has_2fa",
    "interac_daily_limit_cad"
)

fraud_slim = df_fraud.select(
    col("transaction_reference").alias("fraud_txn_ref"),
    "fraud_case_id",
    "fraud_type",
    col("confirmation_status").alias("fraud_confirmation_status"),
    "ml_fraud_score",
    col("severity").alias("fraud_severity")
)

# ── Join all tables ─────────────────────────────────────────
df_gold = (df_transactions
    .join(merchants_slim,   "merchant_id",   "left")
    .join(cardholders_slim, "cardholder_id", "left")
    .join(fraud_slim,
          df_transactions["transaction_id"] == fraud_slim["fraud_txn_ref"],
          "left")
    .withColumn("is_fraud_case",
        when(col("fraud_case_id").isNotNull(), "Y").otherwise("N"))
    .drop("fraud_txn_ref",
          "_pipeline_name", "_batch_date", "_silver_loaded_at")
    .withColumn("_gold_loaded_at", current_timestamp())
    .withColumn("_pipeline_name",  lit(PIPELINE_NAME))
    .withColumn("_batch_date",     lit(BATCH_DATE))
)

print(f"Gold fact rows: {df_gold.count():,}")
print(f"Gold columns  : {len(df_gold.columns)}")

StatementMeta(, 75aa9f49-bfb0-45ba-b525-ffdfe4a8231d, 17, Finished, Available, Finished, False)

Gold fact rows: 145,986
Gold columns  : 51


In [16]:
# ── Write to Gold Lakehouse ─────────────────────────────────
(df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .save(f"{GOLD_PATH}/Tables/dbo/{GOLD_TABLE}"))

# ── Verify ──────────────────────────────────────────────────
final_count = spark.read.format("delta").load(
    f"{GOLD_PATH}/Tables/dbo/{GOLD_TABLE}").count()

fraud_count = df_gold.filter(col("is_fraud_case") == "Y").count()
approved    = df_gold.filter(col("is_approved") == "Y").count()
flagged     = df_gold.filter(col("is_flagged") == "Y").count()

print("\n" + "="*60)
print("GOLD FACT TRANSACTIONS SUMMARY")
print("="*60)
print(f"Total rows          : {final_count:,}")
print(f"Approved txns       : {approved:,}")
print(f"Fraud cases         : {fraud_count:,}")
print(f"Flagged txns        : {flagged:,}")
print(f"Total columns       : {len(df_gold.columns)}")
print(f"Table               : {GOLD_PATH}/Tables/dbo/{GOLD_TABLE}")
print(f"Completed at        : {datetime.now()}")
print("="*60)

StatementMeta(, 75aa9f49-bfb0-45ba-b525-ffdfe4a8231d, 18, Finished, Available, Finished, False)


GOLD FACT TRANSACTIONS SUMMARY
Total rows          : 145,986
Approved txns       : 77,970
Fraud cases         : 2,800
Flagged txns        : 1,548
Total columns       : 51
Table               : abfss://bddf1263-676e-44e7-bf20-6d7b218dc2d2@onelake.dfs.fabric.microsoft.com/07bd3e33-6f0c-4dac-b2a4-b70dca6c6298/Tables/dbo/gold_fact_transactions
Completed at        : 2026-05-06 01:13:01.733453
